# 05b — A hard desert limit: does it improve the backtest?

The backtest of notebook 05 placed the 2025 western front within 12.5 km, but it was **too permissive in arid land**:

- it expected ~9 newly colonised desert cells in 2016–2025, and only 1 appeared;
- it gave a ~40% chance of toad records in the Pilbara by 2025, and none appeared.

The climate models of notebooks 03–04 are correlative and never reach exactly zero, so the colonisation model always keeps a small chance of long jumps into the desert. Mechanistic models take the opposite view: cane toads need standing water to breed and lose water fast through the skin, so true desert should be out of reach (Kearney et al. 2008).

**Variants tested here**, using cells classified as **Köppen desert (BW)** (Peel et al. 2007, notebook 03):

- **Desert as a feature value:** suitability set to zero in desert cells both when training the colonisation model and when simulating.
- **Desert as a mask:** the colonisation model is trained exactly as in notebook 05; only during the simulation is desert suitability set to zero, which drives the colonisation probability of desert cells to practically zero.

Everything else is identical to notebook 05: same features, same logistic regression, same 50-run backtest with data up to 2015, same seeds.

⚠️ A hard limit is a strong assumption: artificial water (dams, troughs) lets toads persist in arid land (Tingley et al. 2013), and 4 desert cells already have toad records.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))
from colonization import FEATURES, cell_year_table, simulate
from environment import BIOCLIM_VARIABLES, environmental_grid
from front import records_to_cells
from native import native_training_cells

PROCESSED = PROJECT_ROOT / "data" / "processed"
CELL_SIZE_M = 25_000
CELL_KM2 = (CELL_SIZE_M / 1000) ** 2
CLIMATE = list(BIOCLIM_VARIABLES.values())
TRAIN_START = 1990
SCENARIOS = ["observed niche", "expanded niche"]
N_RUNS_BACKTEST = 50

cells = environmental_grid(CELL_SIZE_M)
first_detection = records_to_cells(
    pd.read_csv(PROCESSED / "rhinella_marina_clean.csv"), CELL_SIZE_M
)[["cx", "cy", "first_year"]]
cells = cells.merge(first_detection, how="left", on=["cx", "cy"])
cells["first_year"] = cells["first_year"].fillna(np.inf)
cells["x_km"] = (cells["cx"] + 0.5) * CELL_SIZE_M / 1000
cells["y_km"] = (cells["cy"] + 0.5) * CELL_SIZE_M / 1000
native = native_training_cells(CELL_SIZE_M)

desert = (cells["climate_zone"] == "Desert (BW)").to_numpy()
pilbara = (cells["lon"].between(116, 121) & cells["lat"].between(-23, -19)).to_numpy()
west = (cells["lon"] < 141).to_numpy()
x_km = cells["x_km"].to_numpy()
occupied_2015 = (cells["first_year"] <= 2015).to_numpy()
occupied_2025 = (cells["first_year"] <= 2025).to_numpy()
print(f"{desert.sum():,} desert cells ({100 * desert.mean():.0f}% of Australia); "
      f"{(desert & occupied_2025).sum()} of them with toad records by 2025; "
      f"{desert[pilbara].mean():.0%} of the Pilbara box is desert")

5,098 desert cells (41% of Australia); 4 of them with toad records by 2025; 68% of the Pilbara box is desert


## 1. Suitability with and without the desert limit

In [2]:
def climate_model():
    return make_pipeline(StandardScaler(), PolynomialFeatures(degree=2), LogisticRegression(max_iter=5000))


def suitability(scenario: str, cutoff_year: int, desert_limit: bool) -> np.ndarray:
    australia = cells[CLIMATE].assign(toad=(cells["first_year"] <= cutoff_year).astype(int))
    training = australia if scenario == "observed niche" else pd.concat([native[CLIMATE + ["toad"]], australia])
    score = climate_model().fit(training[CLIMATE], training["toad"]).predict_proba(cells[CLIMATE])[:, 1]
    return np.where(desert, 0.0, score) if desert_limit else score


def western_front(occupied: np.ndarray) -> float:
    return np.sort(x_km[occupied & west])[2]


MODES = ["none", "feature", "mask"]
VARIANTS = [(s, mode) for s in SCENARIOS for mode in MODES]
suitability_2015 = {(s, limit): suitability(s, 2015, limit) for s in SCENARIOS for limit in [False, True]}


def scores(scenario: str, mode: str, table: dict):
    '''Suitability used to train the colonisation model and suitability used to simulate.'''
    return table[(scenario, mode == "feature")], table[(scenario, mode != "none")]


for (scenario, limit), score in suitability_2015.items():
    print(f"{scenario:15s} desert limit={limit!s:5s}  mean suitability in desert: {score[desert].mean():.3f}")

observed niche  desert limit=False  mean suitability in desert: 0.003
observed niche  desert limit=True   mean suitability in desert: 0.000
expanded niche  desert limit=False  mean suitability in desert: 0.008
expanded niche  desert limit=True   mean suitability in desert: 0.000


## 2. Backtest: trained with data up to 2015, simulated to 2025

In [3]:
def backtest(key):
    train_score, sim_score = scores(*key, suitability_2015)
    train = cell_year_table(cells, train_score, range(TRAIN_START, 2016))
    test = cell_year_table(cells, sim_score, range(2016, 2026))
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(train[FEATURES], train["colonised"])
    simulated = simulate(model, cells, sim_score, 2015, range(2016, 2026), n_runs=N_RUNS_BACKTEST)
    occupied = simulated <= 2025
    fronts = np.array([western_front(run) for run in occupied])
    new = ~occupied_2015
    return simulated, {
        "one-step AUC": roc_auc_score(test["colonised"], model.predict_proba(test[FEATURES])[:, 1]),
        "multi-step AUC": roc_auc_score(occupied_2025[new], occupied.mean(axis=0)[new]),
        "new cells (median; observed 313)": np.median(occupied.sum(axis=1)) - occupied_2015.sum(),
        "new desert cells (median; observed 1)": np.median(occupied[:, desert & new].sum(axis=1)),
        "western front error 2025 (km)": np.median(fronts) - western_front(occupied_2025),
        "P(Pilbara records by 2025; observed: none)": occupied[:, pilbara].any(axis=1).mean(),
    }


runs, results = {}, {}
for key in VARIANTS:
    runs[key], results[key] = backtest(key)
table = pd.DataFrame(results).T
LABELS = {"none": "no limit (notebook 05)", "feature": "desert as feature value", "mask": "desert as mask"}
table.index = [f"{s}, {LABELS[mode]}" for s, mode in table.index]
table.round(3)

,one-step AUC,multi-step AUC,new cells (median; observed 313),new desert cells (median; observed 1),western front error 2025 (km),P(Pilbara records by 2025; observed: none)
"observed niche, no limit (notebook 05)",0.906,0.829,186.5,9.0,12.5,0.40
"observed niche, desert as feature value",0.908,0.819,192.0,0.0,262.5,0.06
"observed niche, desert as mask",0.908,0.828,178.0,0.0,150.0,0.14
"expanded niche, no limit (notebook 05)",0.904,0.817,219.0,12.0,112.5,0.22
"expanded niche, desert as feature value",0.907,0.827,229.0,4.0,150.0,0.10
"expanded niche, desert as mask",0.906,0.836,209.0,0.0,187.5,0.04


How to read it: the rows without the limit reproduce notebook 05 (same seeds). Front error: positive = the simulated front lags behind the observed one; negative = it overshoots.

A variant is better only if it fixes the arid-land problem **without** breaking what already worked (front position and AUC).

In [4]:
verdict = table[["new desert cells (median; observed 1)", "P(Pilbara records by 2025; observed: none)",
                 "western front error 2025 (km)"]]
verdict.round(2)

,new desert cells (median; observed 1),P(Pilbara records by 2025; observed: none),western front error 2025 (km)
"observed niche, no limit (notebook 05)",9.0,0.40,12.5
"observed niche, desert as feature value",0.0,0.06,262.5
"observed niche, desert as mask",0.0,0.14,150.0
"expanded niche, no limit (notebook 05)",12.0,0.22,112.5
"expanded niche, desert as feature value",4.0,0.10,150.0
"expanded niche, desert as mask",0.0,0.04,187.5


### Where did the simulated invasion go beyond the 2015 front?

Expected number of newly colonised cells by 2025 west of the 2015 western front, by climate zone (mean over simulations), against what actually happened.

In [5]:
beyond_2015_front = west & (x_km < western_front(occupied_2015)) & ~occupied_2015
zone = cells.loc[beyond_2015_front, "climate_zone"].to_numpy()
by_zone = {
    "observed": pd.Series(occupied_2025[beyond_2015_front].astype(float)).groupby(zone).sum(),
    "no limit (notebook 05)": pd.Series((runs[("observed niche", "none")] <= 2025)[:, beyond_2015_front].mean(axis=0)).groupby(zone).sum(),
    "desert as mask": pd.Series((runs[("observed niche", "mask")] <= 2025)[:, beyond_2015_front].mean(axis=0)).groupby(zone).sum(),
}
pd.DataFrame(by_zone).fillna(0).round(1)

,observed,no limit (notebook 05),desert as mask
Desert (BW),0.0,2.2,0.0
"Humid, summer rain",24.0,3.4,3.2
"Humid, winter rain",0.0,0.3,0.2
"Humid, year-round rain",2.0,0.5,0.6
Semi-arid (BS),32.0,4.9,3.3


### What the backtest says

- **The desert limit fixes what it was meant to fix:** no new desert cells (1 observed) and a much lower chance of Pilbara records by 2025 (40% → 4–14%; none observed).
- **Putting the limit in the training data is wrong:** with desert suitability fixed at zero, 5,098 cells become extreme values of the suitability predictor and distort its coefficient everywhere; the front lags by 262 km. A hard constraint belongs in the simulation (the mask), not in a predictor.
- **Even with the mask, the front lags by ~150 km.** The table above shows why: part of the westward advance simulated by notebook 05 happened through desert cells, which did not happen in reality. The 12.5 km front error of notebook 05 was partly right for the wrong reason: errors that compensate each other. Without the desert shortcut, the model advances too slowly through the semi-arid Kimberley, where the real front moved.
- **No variant wins on everything.** The model without limit gets the front position right but is too permissive in arid land; the mask gets arid land right but is too slow. We keep both as bounds: the true behaviour of the front is between them, and the difference is the uncertainty about how hard the desert barrier is (Tingley et al. 2013 found that artificial water makes it leaky).

## 3. Forecast 2026–2035 with the desert limit

With the **mask** version. Saved with the same format as notebook 05 so the dynamic page (notebook 06) can show it as an extra scenario.

In [6]:
N_RUNS_FORECAST = 200
FORECAST_YEARS = range(2026, 2036)
BACKTEST_YEARS = range(2016, 2026)
cell_columns = cells[["cx", "cy", "lon", "lat", "first_year"]].reset_index(drop=True)
summaries, forecast_rows = [], {}
for scenario in SCENARIOS:
    name = f"{scenario.replace(' ', '_')}_desert_limit"
    train_2025, sim_2025 = scores(scenario, "mask", {(scenario, limit): suitability(scenario, 2025, limit) for limit in [False, True]})
    table_2025 = cell_year_table(cells, train_2025, range(TRAIN_START, 2026))
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(table_2025[FEATURES], table_2025["colonised"])
    forecast = simulate(model, cells, sim_2025, 2025, FORECAST_YEARS, n_runs=N_RUNS_FORECAST, seed=1)
    for period, simulated, years in [("backtest", runs[(scenario, "mask")], BACKTEST_YEARS), ("forecast", forecast, FORECAST_YEARS)]:
        probability = pd.DataFrame({f"p_{year}": (simulated <= year).mean(axis=0) for year in years})
        pd.concat([cell_columns, probability], axis=1).to_csv(PROCESSED / f"{period}_{name}.csv", index=False)
        for year in years:
            occupied = simulated <= year
            area = occupied.sum(axis=1) * CELL_KM2
            front = np.array([western_front(run) for run in occupied])
            summaries.append({
                "scenario": f"{scenario}, desert limit", "period": period, "year": year,
                "area_median": np.median(area), "area_p5": np.percentile(area, 5), "area_p95": np.percentile(area, 95),
                "front_x_median": np.median(front), "front_x_p5": np.percentile(front, 5), "front_x_p95": np.percentile(front, 95),
                "p_pilbara": occupied[:, pilbara].any(axis=1).mean(),
            })
    occupied_2030 = forecast <= 2030
    forecast_rows[scenario] = {
        "western front advance by 2030 (km, median)": western_front(occupied_2025) - np.median([western_front(r) for r in occupied_2030]),
        "P(Pilbara records by 2030)": occupied_2030[:, pilbara].any(axis=1).mean(),
        "P(Pilbara records by 2035)": (forecast <= 2035)[:, pilbara].any(axis=1).mean(),
    }
pd.DataFrame(summaries).to_csv(PROCESSED / "simulation_summary_by_year_desert_limit.csv", index=False)
pd.DataFrame(forecast_rows).T.round(2)

,"western front advance by 2030 (km, median)",P(Pilbara records by 2030),P(Pilbara records by 2035)
observed niche,25.0,0.06,0.10
expanded niche,25.0,0.04,0.08


## Takeaways for the forecast

- **The western front slows down in every variant:** about 25 km of advance by 2030 with or without the desert limit. That conclusion of notebook 05 is robust.
- **What changes is the chance of crossing into the Pilbara**, which goes from 15–19% by 2030 (no limit) to 4–6% (desert mask), and from 37–39% to 8–10% by 2035. That range is the honest answer: it depends on how leaky the desert barrier is, which the data up to 2025 cannot tell, because the toad has not reached it yet.
- The dynamic page (notebook 06) shows the four scenarios side by side.